# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Msdff/FlyRankAiAssignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring problem, not classification, clustering, or pure ranking. Here's why, compared against each alternative:

**Not classification**: Classification would mean predicting a fixed category forExample: "will this page decline: yes or no." But my editor's real question isn't binary. Two pages can both be "declining," yet one needs review this week and the other can wait a month. A yes or no label throws away that difference. My editor doesn't need a verdict, they need an order.
**Not clustering**: Clustering would group pages into buckets like "old + low word count" vs. "high traffic + short." That's useful for understanding content, but it doesn't tell an editor which specific page to open first it describes structure, not priority. My lane's whole point is producing an action list, not a taxonomy.
**Not ranking in the strict/relative sense**: Pure ranking (like search-result ranking) compares items pairwise or listwise, optimizing for correct relative order specifically. I don't need pairwise comparisons I need an absolute magnitude per page (how much opportunity does this page represent, on its own) that happens to support sorting. That distinction matters: a score is portable (I can set a threshold, e.g. "review anything above 5%"), while a pure ranking model only tells you order, not "how much better."
**Why scoring fits**: Scoring gives a continuous number per page that it can supports sorting into a queue, or supports threshold-based decisions ("review top 20"), and can be validated against an observed proxy (trend_pct magnitude) using precision@K. It matches exactly what the editor needs: a defensible, explainable priority list not something else.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I don't have a real label that says "an editor picked this page to fix." So instead, I use a stand-in (proxy): opportunity_score = abs(trend_pct).

trend_pct tells us how much a page's traffic went up or down. Taking the absolute value just means I care about the size of the change, not whether it went up or down.

This proxy is good because it's measured from real data, not something I made up or guessed. It's an actual number that happened, not a rule someone wrote by hand.

One important rule: trend_pct and trend_direction can be my target (the thing I'm trying to predict), but they can never be used as inputs (features) to the model. That's because trend_direction is literally calculated from trend_pct, so if the model saw either one as an input, it would basically be cheating by looking at the answer before predicting it.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'll use Precision@K.

Here's what that means: I take my top 20 pages (the ones my model ranks as most needing attention) and check how many of them actually match the top 20 pages when I look at the real, measured traffic change (trend_pct).

I test this on data the model has never seen before, so the check is fair and honest.

Why this metric makes sense: editors can only review a small number of pages each week, say, 20. So the real question isn't "is my model perfect overall," it's "did my model correctly pick out the 20 pages that matter most?" Precision@K answers exactly that question.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(df.shape)

# One row = one content page (pseudonymized), per client
df[["content_id", "client_id", "word_count",
    "avg_position", "ctr", "engagement_rate", "ai_traffic_pct",
    "trend_pct"]].head()

(30000, 44)


,content_id,client_id,word_count,avg_position,ctr,engagement_rate,ai_traffic_pct,trend_pct
0,content_304f48230142,client_f369cb89fc,3221.0,10.6,0.76,5.88,0.0,-41.4
1,content_a1fb4e703a9e,client_4e07408562,2481.0,20.3,0.05,0.00,0.0,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,3515.0,36.5,0.09,0.00,0.0,-60.9
3,content_331d6c4de07b,client_19581e27de,NaN,6.2,0.49,1.28,0.0,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,2803.0,44.0,0.13,0.00,0.0,-34.7


The dataset loaded successfully, 30,000 rows and 44 columns, matching what was expected. The preview table confirms the unit of analysis: each row is one content page, with its own content_id, word_count, avg_position, ctr, and other metrics sitting side by side.

In [4]:
# Build the proxy target column
df["opportunity_score"] = df["trend_pct"].abs()

print((df["avg_position"] == 0).sum(), "rows with avg_position == 0 (means 'no data', not rank zero)")
print(df["ctr"].describe())

df[["content_id", "trend_pct", "opportunity_score"]].sort_values(
    "opportunity_score", ascending=False
).head(10)

1205 rows with avg_position == 0 (means 'no data', not rank zero)
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64


,content_id,trend_pct,opportunity_score
24695,content_dd882c4152ac,44900.0,44900.0
15405,content_a023517539fe,27907.3,27907.3
14549,content_d020d42e7fcc,26166.7,26166.7
3561,content_22f4d2f58c42,21400.0,21400.0
19697,content_4f1966b37335,17077.8,17077.8
24726,content_aac5bd559d85,15825.0,15825.0
9097,content_32ff84795595,14900.0,14900.0
27305,content_ceedad7ba6dc,11650.0,11650.0
1132,content_a6c4ef450727,11266.7,11266.7
19525,content_f6e1f66b051e,11250.0,11250.0


1,205 rows have avg_position == 0,  but no position data available for those pages, not literally "ranked #0." I don't accidentally treat those 1,205 pages as top performers later.
The ctr summary stats confirm the values are small percentages (mean ≈ 0.51, max = 100), matching the expected scale no misreading here.
The top-10 table by opportunity_score shows the pages with the single largest measured traffic swings. A few values here (like 44,900%) are extremely large, which is a flag worth noting honestly: these are likely pages that started from a very small traffic base, where even a small absolute change produces a huge percentage swing. This is a limitation of using trend_pct as a proxy. I should treat these extreme outliers with caution rather than assuming they're the "most important" pages to review, and would want to check the underlying raw traffic numbers before trusting this ranking blindly.

In [5]:
# Does content length relate to traffic volume?
short_pages = df[df["word_count"] < 1200]
long_pages = df[df["word_count"] >= 1200]
print("Avg sessions_90d — short pages:", round(short_pages["sessions_90d"].mean(), 1))
print("Avg sessions_90d — long pages:", round(long_pages["sessions_90d"].mean(), 1))

Avg sessions_90d — short pages: 3.3
Avg sessions_90d — long pages: 42.2


Pages with 1,200+ words average 42.2 sessions over 90 days, versus just 3.3 sessions for shorter pages — roughly a 12x difference. This is a real, measured signal: content length and traffic volume are clearly related in this dataset. It's exactly the kind of pattern that supports using a model instead of a fixed rule — length alone isn't the whole story, but it's one interacting factor worth weighing.

In [6]:
# Does competition level relate to click-through rate?
print(df.groupby("competition_level")["ctr"].mean().round(2))

competition_level
HIGH      0.26
LOW       0.34
MEDIUM    0.24
Name: ctr, dtype: float64


Average click-through rate differs by competition level — LOW competition pages get the highest CTR (0.34), followed by HIGH (0.26), then MEDIUM (0.24) lowest. Interestingly, this isn't a straight line (medium isn't in between low and high) — which is itself evidence that the relationship between competition and performance isn't simple or linear. A basic rule like "high competition = deprioritize" would miss this nuance; a model can learn the actual (messier) pattern instead.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule like "flag any page older than 2 years with under 500 words" sounds easy, but it misses how things actually work together. Competition level, how people find the page (search vs. AI tools), and other page details all move differently across 32 different clients, and each client has its own normal range. A fixed rule would need constant manual adjusting for every single client. It also can't tell the difference between "this page has no keyword data because of its type" and "this page is genuinely doing badly" a fixed rule would treat both the same way and get it wrong.

ML can look at all these signals together and figure out how they interact. A simple if-this-then-that rule can't do that it can only check one or two things at a time.

This score feeds directly into the editor's weekly to-do list the highest-scoring pages get reviewed first.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.